[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/086.%20The%20Capacitated%20Facility%20Location%20Problem%20%28CFLP%29/P86-Tier-7_executed.ipynb)

---

# 86. The Capacitated Facility Location Problem (CFLP)

## Tier 7: Human–AI Partnership

### Purpose and scope

This notebook demonstrates a human–AI decision-support workflow for a retail facility-location problem. The input is a set of candidate facilities and customer markets, including costs, capacities, coordinates, and strategic attributes. The workflow produces candidate facility sets, evaluates them using a demand-ordered greedy assignment, lets a human expert compare and constrain candidates, and reports the resulting decisions and metrics.

The facility choices are generated by transparent heuristics; they are **not** mathematically proven global optima. The notebook does not use a mathematical-programming solver. Its greedy assignment can depend on customer order and can leave demand unserved even when a different assignment might serve more demand. Human preferences and constraints are represented by a small illustrative interface, not by a deployed interactive system.

### Reference CFLP formulation

A standard capacitated facility-location model provides a useful reference for the decisions represented here. Let \(I\) be the set of facilities and \(J\) the set of customers. Let \(y_i \in \{0,1\}\) indicate whether facility \(i\) is open, and let \(x_{ij} \in \{0,1\}\) indicate whether customer \(j\) is assigned to facility \(i\). With fixed opening cost \(f_i\), demand \(d_j\), capacity \(K_i\), and unit transportation cost \(c_{ij}\), the reference model is:

$$\begin{aligned}
\min_{x,y} \quad & \sum_{i\in I} f_i y_i + \sum_{i\in I}\sum_{j\in J} d_j c_{ij}x_{ij} \\
\text{subject to} \quad
& \sum_{i\in I}x_{ij}=1 && j\in J,\\
& \sum_{j\in J}d_jx_{ij}\le K_i y_i && i\in I,\\
& x_{ij}\in\{0,1\},\quad y_i\in\{0,1\} && i\in I,\ j\in J.
\end{aligned}$$

Here, the first constraint assigns every customer exactly once; the second limits each open facility's assigned demand to its capacity and prevents assignments to closed facilities. In this notebook, `fixed_cost` corresponds to \(f_i\), `capacity` to \(K_i\), `demand` to \(d_j\), and the distance-based transportation rate to \(c_{ij}\). The code does not solve this reference formulation: its evaluator uses a greedy assignment and reports unmet demand, while proposal-building heuristics aim for specified service levels. Thus, reported candidate costs and service levels describe those heuristics, not a proven optimum of the reference model.

### Partnership workflow

1. **AI proposes:** Four transparent heuristics generate candidates focused on cost, strategic importance, balance, or risk.
2. **Human evaluates:** A preference-based score and qualitative checks make trade-offs visible.
3. **Human constrains:** The expert may specify facility-selection rules based on the chosen candidate.
4. **AI adapts:** The same proposal heuristics apply those rules and re-evaluate the resulting facility sets.
5. **Collaborative decision:** The human selects among the adapted candidates; a simple score summarizes selected attributes.
6. **Learning demonstration:** Preferences are explicitly updated between decision cycles. This is a simulated change in human preferences, not autonomous learning by the AI.

### Modeling assumptions

- Facility and customer coordinates use a common planar coordinate system. Euclidean distance is used as a proxy for transportation distance; the coordinate units are not calibrated to a particular geography.
- Transportation cost is distance multiplied by a rate of 9.5 cost units per demand unit per distance unit. Facility fixed costs and transportation costs use the same cost unit.
- Customers are indivisible in the assignment heuristic: each customer is assigned in full to one facility or left unserved. A facility's capacity is a limit on total demand assigned to it.
- Strategic scores and risk scores are descriptive indicators defined by the notebook, not measured financial or operational outcomes.
- The example does not model facility-opening uncertainty, transportation capacity, service-time limits, multiple products, or demand uncertainty.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Optional, Set, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class Facility:
    """Candidate facility and its operational and strategic attributes."""

    id: int
    name: str
    x_coord: float
    y_coord: float
    fixed_cost: float
    capacity: float
    strategic_importance: float
    expansion_potential: float
    risk_profile: str
    market_coverage: str


@dataclass
class Customer:
    """Customer market with demand and strategic attributes."""

    id: int
    name: str
    x_coord: float
    y_coord: float
    demand: float
    strategic_value: float
    growth_prospect: float
    competitive_intensity: float


@dataclass
class AIProposal:
    """A candidate facility set with metrics and an explanation."""

    id: int
    name: str
    open_facilities: Set[int]
    total_cost: float
    service_level: float
    risk_score: float
    strategic_alignment: float
    reasoning: List[str]
    confidence: float


@dataclass
class HumanConstraint:
    """A human-specified rule for facility selection."""

    type: str
    description: str
    facility_ids: Set[int]
    value: Optional[float]
    priority: int


@dataclass
class CollaborativeDecision:
    """Summary of the selected human–AI decision."""

    selected_proposal: AIProposal
    human_modifications: List[str]
    ai_adaptations: List[str]
    final_cost: float
    collaboration_score: float
    learning_insights: List[str]

In [ ]:
def initialize_facilities() -> List[Facility]:
    """Create the six candidate facilities used in the example."""
    return [
        Facility(1, "North Strategic Hub", 20, 80, 180000, 1500,
                 0.9, 0.8, "LOW", "URBAN"),
        Facility(2, "Central DC", 50, 50, 140000, 1200,
                 0.7, 0.6, "MEDIUM", "SUBURBAN"),
        Facility(3, "South Growth Center", 80, 20, 100000, 800,
                 0.8, 0.9, "HIGH", "RURAL"),
        Facility(4, "East Market Entry", 85, 65, 120000, 1000,
                 0.6, 0.7, "MEDIUM", "SUBURBAN"),
        Facility(5, "West Logistics Hub", 15, 35, 110000, 900,
                 0.5, 0.5, "LOW", "RURAL"),
        Facility(6, "Northeast Premium", 70, 85, 160000, 1300,
                 0.9, 0.4, "LOW", "URBAN"),
    ]


def initialize_customers() -> List[Customer]:
    """Create the eight customer markets used in the example."""
    return [
        Customer(1, "Flagship Store A", 25, 75, 500, 0.9, 0.7, 0.8),
        Customer(2, "Mega Store B", 45, 65, 600, 0.8, 0.6, 0.9),
        Customer(3, "Growth Store C", 75, 35, 350, 0.7, 0.9, 0.4),
        Customer(4, "Premium Store D", 90, 60, 450, 0.8, 0.5, 0.7),
        Customer(5, "Community Store E", 30, 20, 250, 0.5, 0.6, 0.3),
        Customer(6, "Emerging Market F", 60, 90, 400, 0.7, 0.8, 0.6),
        Customer(7, "Rural Outlet G", 85, 40, 200, 0.4, 0.7, 0.2),
        Customer(8, "Suburban Store H", 55, 75, 380, 0.6, 0.5, 0.5),
    ]


facilities = initialize_facilities()
customers = initialize_customers()

print("Facility-location example initialized.")
print(f"Candidate facilities: {len(facilities)}")
print(f"Customer markets: {len(customers)}")
print("Strategic dimensions include risk, growth, coverage, and competition.")

In [ ]:
class ExplainableAIOptimizer:
    """Generate and evaluate heuristic facility-location proposals."""

    def __init__(self, facilities: List[Facility], customers: List[Customer]):
        self.facilities = facilities
        self.customers = customers
        self.facility_by_id = {facility.id: facility for facility in facilities}
        self.customer_by_id = {customer.id: customer for customer in customers}
        self.transport_costs = self._calculate_transport_costs()
        self.learning_history = []

    def _calculate_transport_costs(self) -> Dict[Tuple[int, int], float]:
        """Compute unit transportation cost from Euclidean distance."""
        costs = {}
        for facility in self.facilities:
            for customer in self.customers:
                distance = np.hypot(
                    facility.x_coord - customer.x_coord,
                    facility.y_coord - customer.y_coord,
                )
                costs[(facility.id, customer.id)] = distance * 9.5
        return costs

    def evaluate_solution(self, open_facilities: Set[int]) -> Dict:
        """Greedily assign larger-demand customers first and report service."""
        unknown_ids = open_facilities - self.facility_by_id.keys()
        if unknown_ids:
            raise ValueError(f"Unknown facility IDs: {sorted(unknown_ids)}")

        fixed_cost = sum(
            self.facility_by_id[facility_id].fixed_cost
            for facility_id in open_facilities
        )
        facility_loads = {facility_id: 0.0 for facility_id in open_facilities}
        assignments = {}
        transport_cost = 0.0
        unmet_demand = 0.0

        # Stable ordering makes equal-demand and equal-cost decisions reproducible.
        sorted_customers = sorted(
            self.customers, key=lambda customer: (-customer.demand, customer.id)
        )
        for customer in sorted_customers:
            eligible_facilities = [
                facility_id
                for facility_id in sorted(open_facilities)
                if facility_loads[facility_id] + customer.demand
                <= self.facility_by_id[facility_id].capacity
            ]

            if not eligible_facilities:
                unmet_demand += customer.demand
                continue

            best_facility = min(
                eligible_facilities,
                key=lambda facility_id: (
                    self.transport_costs[(facility_id, customer.id)], facility_id
                ),
            )
            assignments[customer.id] = best_facility
            facility_loads[best_facility] += customer.demand
            transport_cost += (
                self.transport_costs[(best_facility, customer.id)] * customer.demand
            )

        total_demand = sum(customer.demand for customer in self.customers)
        service_level = (
            (total_demand - unmet_demand) / total_demand if total_demand else 0.0
        )
        return {
            "total_cost": fixed_cost + transport_cost,
            "fixed_cost": fixed_cost,
            "transport_cost": transport_cost,
            "service_level": service_level,
            "assignments": assignments,
            "facility_loads": facility_loads,
            "unmet_demand": unmet_demand,
        }

    def calculate_strategic_metrics(self, open_facilities: Set[int]) -> Dict:
        """Summarize strategic attributes for the selected facility set."""
        selected = [
            self.facility_by_id[facility_id]
            for facility_id in sorted(open_facilities)
        ]
        count = len(selected)
        strategic_importance = (
            sum(facility.strategic_importance for facility in selected) / count
            if count else 0.0
        )
        expansion_potential = (
            sum(facility.expansion_potential for facility in selected) / count
            if count else 0.0
        )

        risk_scores = {"LOW": 0.1, "MEDIUM": 0.5, "HIGH": 0.9}
        importance_total = sum(
            facility.strategic_importance for facility in selected
        )
        risk_score = (
            sum(
                risk_scores[facility.risk_profile]
                * facility.strategic_importance
                for facility in selected
            ) / importance_total
            if importance_total else 0.0
        )
        coverage_types = {facility.market_coverage for facility in selected}
        coverage_balance = len(coverage_types) / 3.0

        evaluation = self.evaluate_solution(open_facilities)
        served_customers = set(evaluation["assignments"])
        strategic_coverage = (
            sum(
                customer.strategic_value
                for customer in self.customers
                if customer.id in served_customers
            ) / len(self.customers)
            if self.customers else 0.0
        )
        return {
            "strategic_importance": strategic_importance,
            "expansion_potential": expansion_potential,
            "risk_score": risk_score,
            "coverage_balance": coverage_balance,
            "strategic_coverage": strategic_coverage,
        }

    def generate_proposals(
        self, constraints: Optional[List[HumanConstraint]] = None
    ) -> List[AIProposal]:
        """Build four candidates using the notebook's selection heuristics."""
        return [
            self._generate_cost_optimized_proposal(constraints),
            self._generate_strategic_proposal(constraints),
            self._generate_balanced_proposal(constraints),
            self._generate_risk_averse_proposal(constraints),
        ]

    def _make_proposal(
        self,
        proposal_id: int,
        name: str,
        open_facilities: Set[int],
        reasoning: List[str],
        confidence: float,
    ) -> AIProposal:
        """Evaluate a facility set and package its reported metrics."""
        evaluation = self.evaluate_solution(open_facilities)
        strategic = self.calculate_strategic_metrics(open_facilities)
        return AIProposal(
            id=proposal_id,
            name=name,
            open_facilities=set(open_facilities),
            total_cost=evaluation["total_cost"],
            service_level=evaluation["service_level"],
            risk_score=strategic["risk_score"],
            strategic_alignment=strategic["strategic_importance"],
            reasoning=reasoning,
            confidence=confidence,
        )

    def _build_by_order(
        self,
        ordered_facility_ids: List[int],
        target_service: float,
        constraints: Optional[List[HumanConstraint]],
    ) -> Set[int]:
        """Add facilities in a stated order until the service target is met."""
        open_facilities = set()
        evaluation = self.evaluate_solution(open_facilities)
        for facility_id in ordered_facility_ids:
            if evaluation["service_level"] >= target_service:
                break
            open_facilities.add(facility_id)
            evaluation = self.evaluate_solution(open_facilities)
        if constraints:
            open_facilities = self._apply_constraints(open_facilities, constraints)
        return open_facilities

    def _generate_cost_optimized_proposal(
        self, constraints: Optional[List[HumanConstraint]]
    ) -> AIProposal:
        """Prefer low fixed-cost facilities, targeting 95% service."""
        ordered_ids = [
            facility.id
            for facility in sorted(
                self.facilities, key=lambda item: (item.fixed_cost, item.id)
            )
        ]
        selected = self._build_by_order(ordered_ids, 0.95, constraints)
        evaluation = self.evaluate_solution(selected)
        return self._make_proposal(
            1, "Cost Optimized", selected,
            [
                "Facilities were considered in ascending fixed-cost order.",
                f"The resulting greedy assignment serves {evaluation['service_level']:.1%} of demand.",
            ],
            0.90,
        )

    def _generate_strategic_proposal(
        self, constraints: Optional[List[HumanConstraint]]
    ) -> AIProposal:
        """Prefer strategically important facilities, targeting 90% service."""
        ordered_ids = [
            facility.id
            for facility in sorted(
                self.facilities,
                key=lambda item: (-item.strategic_importance, item.id),
            )
        ]
        selected = self._build_by_order(ordered_ids, 0.90, constraints)
        strategic = self.calculate_strategic_metrics(selected)
        return self._make_proposal(
            2, "Strategic Alignment", selected,
            [
                f"Facilities were prioritized by strategic importance; mean importance is {strategic['strategic_importance']:.2f}.",
                "The selection order emphasizes strategic positioning rather than minimum cost.",
            ],
            0.85,
        )

    def _generate_balanced_proposal(
        self, constraints: Optional[List[HumanConstraint]]
    ) -> AIProposal:
        """Rank facilities by an equal-weight cost and strategy score."""
        max_cost = max(facility.fixed_cost for facility in self.facilities)
        facility_scores = []
        for facility in self.facilities:
            cost_score = 1.0 - facility.fixed_cost / max_cost
            combined_score = (
                0.5 * cost_score + 0.5 * facility.strategic_importance
            )
            facility_scores.append((facility.id, combined_score))
        ordered_ids = [
            facility_id
            for facility_id, _ in sorted(
                facility_scores, key=lambda item: (-item[1], item[0])
            )
        ]
        selected = self._build_by_order(ordered_ids, 0.93, constraints)
        evaluation = self.evaluate_solution(selected)
        return self._make_proposal(
            3, "Balanced Approach", selected,
            [
                "Facilities were ranked by an equal-weight combination of normalized fixed-cost attractiveness and strategic importance.",
                f"The resulting greedy assignment serves {evaluation['service_level']:.1%} of demand.",
            ],
            0.88,
        )

    def _generate_risk_averse_proposal(
        self, constraints: Optional[List[HumanConstraint]]
    ) -> AIProposal:
        """Add low-risk facilities first, then medium-risk facilities if needed."""
        low_risk = sorted(
            (facility for facility in self.facilities
             if facility.risk_profile == "LOW"),
            key=lambda item: item.id,
        )
        medium_risk = sorted(
            (facility for facility in self.facilities
             if facility.risk_profile == "MEDIUM"),
            key=lambda item: item.id,
        )
        selected = set()
        for facility in low_risk + medium_risk:
            if self.evaluate_solution(selected)["service_level"] >= 0.90:
                break
            selected.add(facility.id)
        if constraints:
            selected = self._apply_constraints(selected, constraints)
        strategic = self.calculate_strategic_metrics(selected)
        return self._make_proposal(
            4, "Risk Averse", selected,
            [
                f"Low-risk facilities are considered before medium-risk facilities; resulting risk score is {strategic['risk_score']:.2f}.",
                "High-risk facilities are not added by this proposal heuristic.",
            ],
            0.92,
        )

    def _apply_constraints(
        self,
        open_facilities: Set[int],
        constraints: List[HumanConstraint],
    ) -> Set[int]:
        """Apply supported facility rules in ascending priority order.

        Higher-priority rules are applied later and therefore take precedence
        when rules conflict. A resulting set is re-evaluated for service; the
        constraints do not guarantee a particular service level.
        """
        selected = set(open_facilities)
        for constraint in sorted(constraints, key=lambda item: item.priority):
            if constraint.type == "MUST_OPEN":
                selected.update(constraint.facility_ids)
            elif constraint.type == "CANNOT_OPEN":
                selected.difference_update(constraint.facility_ids)
            elif constraint.type == "MIN_FACILITIES":
                target = int(constraint.value or 0)
                candidates = sorted(
                    (facility for facility in self.facilities
                     if facility.id not in selected),
                    key=lambda item: (item.fixed_cost, item.id),
                )
                for facility in candidates[:max(0, target - len(selected))]:
                    selected.add(facility.id)
            elif constraint.type == "MAX_FACILITIES":
                target = int(constraint.value or 0)
                ranked = sorted(
                    (self.facility_by_id[facility_id]
                     for facility_id in selected),
                    key=lambda item: (-item.strategic_importance, item.id),
                )
                selected = {facility.id for facility in ranked[:target]}
        return selected


print("Heuristic proposal and evaluation methods are ready.")

In [ ]:
class HumanExpertInterface:
    """Represent human preferences, evaluations, and facility constraints."""

    def __init__(self, facilities: List[Facility]):
        self.facilities = facilities
        self.expertise_areas = [
            "Strategic Planning",
            "Market Analysis",
            "Risk Management",
            "Operational Excellence",
            "Financial Analysis",
        ]
        self.decision_history = []
        self.preferences = {
            "cost_sensitivity": 0.7,
            "risk_tolerance": 0.6,
            "strategic_focus": 0.8,
            "expansion_priority": 0.5,
        }

    def evaluate_proposals(self, proposals: List[AIProposal]) -> Dict:
        """Score candidates using stated preferences and report qualitative checks."""
        if not proposals:
            return {}
        maximum_cost = max(proposal.total_cost for proposal in proposals)
        evaluations = {}
        facility_by_id = {facility.id: facility for facility in self.facilities}

        for proposal in proposals:
            normalized_cost = (
                proposal.total_cost / maximum_cost if maximum_cost > 0 else 0.0
            )
            expansion = (
                sum(
                    facility_by_id[facility_id].expansion_potential
                    for facility_id in proposal.open_facilities
                ) / len(proposal.open_facilities)
                if proposal.open_facilities else 0.0
            )
            # Each term is a preference-weighted utility: higher scores are preferred.
            human_score = (
                self.preferences["cost_sensitivity"] * (1.0 - normalized_cost)
                + (1.0 - self.preferences["risk_tolerance"])
                * (1.0 - proposal.risk_score)
                + self.preferences["strategic_focus"]
                * proposal.strategic_alignment
                + self.preferences["expansion_priority"] * expansion
            )
            evaluations[proposal.id] = {
                "human_score": human_score,
                "insights": self._generate_human_insights(proposal),
                "concerns": self._identify_concerns(proposal),
                "recommendations": self._generate_recommendations(proposal),
            }
        return evaluations

    def _generate_human_insights(self, proposal: AIProposal) -> List[str]:
        insights = []
        if proposal.strategic_alignment > 0.8:
            insights.append("Strong strategic alignment with long-term goals")
        elif proposal.strategic_alignment < 0.6:
            insights.append("Limited strategic value; consider the business context")
        if proposal.risk_score > 0.7:
            insights.append("High risk profile may require mitigation plans")
        elif proposal.risk_score < 0.3:
            insights.append("Low risk score supports operational stability")
        if proposal.service_level < 0.9:
            insights.append("Service level is below the 90% reference threshold")
        elif proposal.service_level > 0.95:
            insights.append("High service level supports demanding service expectations")
        return insights

    def _identify_concerns(self, proposal: AIProposal) -> List[str]:
        concerns = []
        if proposal.service_level < 0.85:
            concerns.append("Unserved demand may affect customer satisfaction")
        if len(proposal.open_facilities) < 3:
            concerns.append("A small network may limit geographic coverage")
        elif len(proposal.open_facilities) > 5:
            concerns.append("A large network may increase fixed-cost exposure")
        if proposal.risk_score > 0.6:
            concerns.append("Operational risk may affect business continuity")
        return concerns

    def _generate_recommendations(self, proposal: AIProposal) -> List[str]:
        recommendations = []
        if proposal.service_level < 0.9:
            recommendations.append("Review capacity and assignment coverage")
        if proposal.risk_score > 0.5:
            recommendations.append("Consider contingency plans for selected facilities")
        if proposal.strategic_alignment < 0.7:
            recommendations.append("Review the strategic fit of the selected facilities")
        return recommendations

    def add_constraints(self, proposal: AIProposal) -> List[HumanConstraint]:
        """Create example constraints from the selected proposal's reported metrics."""
        constraints = []
        if proposal.total_cost > 550000:
            constraints.append(
                HumanConstraint(
                    "MAX_FACILITIES",
                    "Limit the number of facilities considered for budget discipline",
                    set(), 4, 8,
                )
            )
        if proposal.risk_score > 0.6:
            high_risk_ids = {
                facility.id for facility in self.facilities
                if facility.risk_profile == "HIGH"
            }
            if high_risk_ids:
                constraints.append(
                    HumanConstraint(
                        "CANNOT_OPEN",
                        "Avoid high-risk facilities due to operational concerns",
                        high_risk_ids, None, 9,
                    )
                )
        if proposal.strategic_alignment < 0.7:
            high_importance_ids = [
                facility.id for facility in sorted(
                    self.facilities,
                    key=lambda item: (-item.strategic_importance, item.id),
                ) if facility.strategic_importance > 0.8
            ]
            if high_importance_ids:
                constraints.append(
                    HumanConstraint(
                        "MUST_OPEN",
                        "Include facilities with high strategic importance",
                        set(high_importance_ids[:2]), None, 7,
                    )
                )
        return constraints


print("Human preference and constraint methods are ready.")

In [ ]:
print("=" * 76)
print("HUMAN–AI DECISION-SUPPORT INITIALIZATION")
print("=" * 76)

ai_optimizer = ExplainableAIOptimizer(facilities, customers)
human_expert = HumanExpertInterface(facilities)

print(
    f"Heuristic evaluator: {len(facilities)} facilities and "
    f"{len(customers)} customer markets"
)
print(f"Human expertise areas: {len(human_expert.expertise_areas)}")
print(
    "Initial preferences: "
    f"cost sensitivity={human_expert.preferences['cost_sensitivity']:.1f}, "
    f"risk tolerance={human_expert.preferences['risk_tolerance']:.1f}, "
    f"strategic focus={human_expert.preferences['strategic_focus']:.1f}, "
    f"expansion priority={human_expert.preferences['expansion_priority']:.1f}"
)

print("\n" + "=" * 60)
print("DECISION CYCLE 1: INITIAL AI PROPOSALS")
print("=" * 60)
initial_proposals = ai_optimizer.generate_proposals()

for proposal in initial_proposals:
    print(f"\n{proposal.name} (ID {proposal.id})")
    print(f"  Evaluated cost: ${proposal.total_cost:,.2f}")
    print(f"  Service level: {proposal.service_level:.1%}")
    print(f"  Risk score: {proposal.risk_score:.2f}")
    print(f"  Strategic importance: {proposal.strategic_alignment:.2f}")
    print(f"  Open facilities: {sorted(proposal.open_facilities)}")
    print(f"  Rationale: {' '.join(proposal.reasoning)}")
    print(f"  Heuristic confidence indicator: {proposal.confidence:.1%}")

In [ ]:
print("\n" + "=" * 60)
print("HUMAN EVALUATION AND FEEDBACK")
print("=" * 60)

human_evaluations = human_expert.evaluate_proposals(initial_proposals)
for proposal in initial_proposals:
    evaluation = human_evaluations[proposal.id]
    print(f"\n{proposal.name}")
    print(f"  Preference-based score: {evaluation['human_score']:.3f}")
    if evaluation["insights"]:
        print(f"  Insights: {'; '.join(evaluation['insights'])}")
    if evaluation["concerns"]:
        print(f"  Concerns: {'; '.join(evaluation['concerns'])}")
    if evaluation["recommendations"]:
        print(f"  Recommendations: {'; '.join(evaluation['recommendations'])}")

best_proposal = max(
    initial_proposals,
    key=lambda proposal: human_evaluations[proposal.id]["human_score"],
)
human_constraints = human_expert.add_constraints(best_proposal)
print(f"\nHuman-selected candidate: {best_proposal.name}")
print(f"Constraints specified: {len(human_constraints)}")
for constraint in human_constraints:
    print(f"  - {constraint.type}: {constraint.description}")

print("\n" + "=" * 60)
print("AI RESPONSE TO HUMAN CONSTRAINTS")
print("=" * 60)
adapted_proposals = ai_optimizer.generate_proposals(human_constraints)
adapted_evaluations = human_expert.evaluate_proposals(adapted_proposals)

for proposal in adapted_proposals:
    print(f"\n{proposal.name} (ID {proposal.id})")
    print(f"  Evaluated cost: ${proposal.total_cost:,.2f}")
    print(f"  Service level: {proposal.service_level:.1%}")
    print(f"  Open facilities: {sorted(proposal.open_facilities)}")
    print(f"  Applied constraint count: {len(human_constraints)}")
    print(f"  Rationale: {' '.join(proposal.reasoning)}")

best_adapted_proposal = max(
    adapted_proposals,
    key=lambda proposal: adapted_evaluations[proposal.id]["human_score"],
)
score_change = (
    adapted_evaluations[best_adapted_proposal.id]["human_score"]
    - human_evaluations[best_proposal.id]["human_score"]
)
print(f"\nHuman-selected adapted candidate: {best_adapted_proposal.name}")
print(f"Change in preference-based score: {score_change:+.3f}")
print(
    "A constraint can be redundant for a candidate; compare facility sets and "
    "service levels to determine whether it changed that candidate."
)

In [ ]:
def make_collaborative_decision(
    ai_proposal: AIProposal,
    human_evaluations: Dict,
    constraints: List[HumanConstraint],
) -> CollaborativeDecision:
    """Record the selected candidate and a transparent attribute-based score."""
    if ai_proposal.id not in human_evaluations:
        raise KeyError("The selected proposal must have a human evaluation.")

    human_score = human_evaluations[ai_proposal.id]["human_score"]
    human_modifications = [
        f"Selected {ai_proposal.name} using the preference-based evaluation",
        f"Specified {len(constraints)} facility-selection constraints",
        "Reviewed cost, service, risk, and strategic attributes",
    ]
    ai_adaptations = [
        "Applied the supplied rules within the proposal heuristics",
        f"Reported a greedy-assignment service level of {ai_proposal.service_level:.1%}",
        "Re-evaluated cost and strategic metrics for the resulting facility set",
    ]
    collaboration_score = (
        0.4 * ai_proposal.confidence
        + 0.4 * (1.0 - ai_proposal.risk_score)
        + 0.2 * ai_proposal.strategic_alignment
    )
    learning_insights = [
        f"The human preference score for the selected candidate is {human_score:.3f}.",
        "Facility constraints express business rules that are not part of the cost calculation.",
        "The collaboration score is a descriptive index, not a measured improvement in solution quality.",
    ]
    return CollaborativeDecision(
        selected_proposal=ai_proposal,
        human_modifications=human_modifications,
        ai_adaptations=ai_adaptations,
        final_cost=ai_proposal.total_cost,
        collaboration_score=collaboration_score,
        learning_insights=learning_insights,
    )


decision = make_collaborative_decision(
    best_adapted_proposal,
    adapted_evaluations,
    human_constraints,
)

print("\n" + "=" * 76)
print("COLLABORATIVE DECISION")
print("=" * 76)
print(f"Selected candidate: {decision.selected_proposal.name}")
print(f"Evaluated cost: ${decision.final_cost:,.2f}")
print(f"Descriptive collaboration score: {decision.collaboration_score:.3f}")
print(f"Open facilities: {sorted(decision.selected_proposal.open_facilities)}")
print("\nHuman input:")
for item in decision.human_modifications:
    print(f"  - {item}")
print("AI response:")
for item in decision.ai_adaptations:
    print(f"  - {item}")
print("Interpretation:")
for item in decision.learning_insights:
    print(f"  - {item}")

In [ ]:
print("\n" + "=" * 76)
print("MULTIPLE DECISION CYCLES: PREFERENCE UPDATES")
print("=" * 76)

# Store the initial values so the preference changes are explicit and auditable.
initial_preferences = human_expert.preferences.copy()
human_expert.preferences["cost_sensitivity"] = 0.6
human_expert.preferences["strategic_focus"] = 0.9
human_expert.preferences["risk_tolerance"] = 0.4

print("Cycle 2 preferences (initial value -> updated value):")
for name in ("cost_sensitivity", "strategic_focus", "risk_tolerance"):
    print(
        f"  {name}: {initial_preferences[name]:.1f} -> "
        f"{human_expert.preferences[name]:.1f}"
    )

cycle2_proposals = ai_optimizer.generate_proposals()
cycle2_evaluations = human_expert.evaluate_proposals(cycle2_proposals)
cycle2_best = max(
    cycle2_proposals,
    key=lambda proposal: cycle2_evaluations[proposal.id]["human_score"],
)
print(f"\nCycle 2 human-selected candidate: {cycle2_best.name}")
print(f"Evaluated cost: ${cycle2_best.total_cost:,.2f}")
print(f"Strategic importance: {cycle2_best.strategic_alignment:.2f}")
print(f"Risk score: {cycle2_best.risk_score:.2f}")

print("\nCycle 3: expertise-area demonstration")
original_expertise = human_expert.expertise_areas.copy()
human_expert.expertise_areas.extend(["Advanced Analytics", "Scenario Planning"])
print(f"Expertise areas before: {original_expertise}")
print(f"Expertise areas after: {human_expert.expertise_areas}")
print(
    "The added labels document a scenario; they do not alter the scoring "
    "formula or automatically change the human preferences."
)

cycle3_proposals = ai_optimizer.generate_proposals()
cycle3_evaluations = human_expert.evaluate_proposals(cycle3_proposals)
cycle3_best = max(
    cycle3_proposals,
    key=lambda proposal: cycle3_evaluations[proposal.id]["human_score"],
)
print(f"\nCycle 3 human-selected candidate: {cycle3_best.name}")
print(f"Evaluated cost: ${cycle3_best.total_cost:,.2f}")
print(f"Strategic importance: {cycle3_best.strategic_alignment:.2f}")
print(f"Risk score: {cycle3_best.risk_score:.2f}")

cycles_data = [
    {
        "cycle": 1,
        "proposal": best_proposal.name,
        "cost": best_proposal.total_cost,
        "strategic_alignment": best_proposal.strategic_alignment,
        "risk_score": best_proposal.risk_score,
        "service_level": best_proposal.service_level,
        "human_score": human_evaluations[best_proposal.id]["human_score"],
    },
    {
        "cycle": 2,
        "proposal": cycle2_best.name,
        "cost": cycle2_best.total_cost,
        "strategic_alignment": cycle2_best.strategic_alignment,
        "risk_score": cycle2_best.risk_score,
        "service_level": cycle2_best.service_level,
        "human_score": cycle2_evaluations[cycle2_best.id]["human_score"],
    },
    {
        "cycle": 3,
        "proposal": cycle3_best.name,
        "cost": cycle3_best.total_cost,
        "strategic_alignment": cycle3_best.strategic_alignment,
        "risk_score": cycle3_best.risk_score,
        "service_level": cycle3_best.service_level,
        "human_score": cycle3_evaluations[cycle3_best.id]["human_score"],
    },
]
learning_df = pd.DataFrame(cycles_data)
print("\nCycle summary (computed from the candidates):")
print(learning_df.round(3).to_string(index=False))

print(
    "\nInterpretation: cycle-to-cycle changes reflect the stated preference "
    "updates and candidate ranking. They do not establish that the network "
    "became globally optimal or that an autonomous learning process occurred."
)

In [ ]:
# The panels use metrics calculated by the proposal evaluator and human scoring rule.
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle("Human–AI Facility-Location Decision Support", fontsize=16)

cycle_labels = [f"Cycle {row['cycle']}" for row in cycles_data]

axes[0, 0].bar(cycle_labels, [row["cost"] for row in cycles_data], color="steelblue")
axes[0, 0].set_title("Evaluated Candidate Cost")
axes[0, 0].set_ylabel("Cost units")
axes[0, 0].grid(axis="y", alpha=0.3)

axes[0, 1].plot(
    cycle_labels,
    [row["service_level"] for row in cycles_data],
    marker="o",
    color="darkgreen",
)
axes[0, 1].set_title("Greedy-Assignment Service Level")
axes[0, 1].set_ylabel("Share of total demand served")
axes[0, 1].set_ylim(0, 1.05)
axes[0, 1].grid(alpha=0.3)

axes[0, 2].plot(
    cycle_labels,
    [row["human_score"] for row in cycles_data],
    marker="s",
    color="darkorange",
)
axes[0, 2].set_title("Preference-Based Human Score")
axes[0, 2].set_ylabel("Score (higher is preferred)")
axes[0, 2].grid(alpha=0.3)

axes[1, 0].plot(
    cycle_labels,
    [row["strategic_alignment"] for row in cycles_data],
    marker="o",
    label="Mean facility strategic importance",
)
axes[1, 0].plot(
    cycle_labels,
    [row["risk_score"] for row in cycles_data],
    marker="s",
    label="Importance-weighted risk score",
)
axes[1, 0].set_title("Strategic and Risk Indicators")
axes[1, 0].set_ylabel("Score")
axes[1, 0].set_ylim(0, 1.05)
axes[1, 0].legend(fontsize=8)
axes[1, 0].grid(alpha=0.3)

# Show the cycle-3 candidate on the supplied coordinate plane.
ax = axes[1, 1]
selected_ids = cycle3_best.open_facilities
for facility in facilities:
    selected = facility.id in selected_ids
    ax.scatter(
        facility.x_coord,
        facility.y_coord,
        marker="s",
        s=180 if selected else 90,
        color="crimson" if selected else "lightgray",
        edgecolor="black",
        label="Open facility" if selected and "Open facility" not in ax.get_legend_handles_labels()[1] else None,
    )
    ax.annotate(
        f"F{facility.id}: {facility.name}",
        (facility.x_coord, facility.y_coord),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=7,
    )
for customer in customers:
    ax.scatter(
        customer.x_coord,
        customer.y_coord,
        marker="^",
        s=75,
        color="royalblue",
        edgecolor="black",
        label="Customer market" if customer.id == customers[0].id else None,
    )
    ax.annotate(
        f"C{customer.id}",
        (customer.x_coord, customer.y_coord),
        xytext=(3, -10),
        textcoords="offset points",
        fontsize=7,
    )
ax.set_title(f"Cycle 3 Candidate: {cycle3_best.name}")
ax.set_xlabel("X coordinate (model units)")
ax.set_ylabel("Y coordinate (model units)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

axes[1, 2].axis("off")
axes[1, 2].text(
    0.02,
    0.98,
    "How to read the comparison\n\n"
    "• Cost includes fixed cost for open facilities and transportation cost for served demand.\n"
    "• Service level is the fraction of total demand assigned by the greedy procedure.\n"
    "• The preference-based score ranks candidates under the current human preferences.\n"
    "• These are heuristic candidate metrics, not proof of a global optimum.",
    va="top",
    fontsize=10,
    wrap=True,
)

fig.tight_layout(rect=(0, 0, 1, 0.96))
plt.show()

In [ ]:
final_solution = ai_optimizer.evaluate_solution(cycle3_best.open_facilities)
final_strategic = ai_optimizer.calculate_strategic_metrics(
    cycle3_best.open_facilities
)

assert set(final_solution["assignments"].values()).issubset(
    cycle3_best.open_facilities
)
assert all(
    final_solution["facility_loads"][facility_id]
    <= ai_optimizer.facility_by_id[facility_id].capacity
    for facility_id in cycle3_best.open_facilities
)
assert np.isclose(
    final_solution["fixed_cost"] + final_solution["transport_cost"],
    final_solution["total_cost"],
)
assert np.isclose(final_solution["total_cost"], cycle3_best.total_cost)

print("\n" + "=" * 76)
print("CYCLE 3 CANDIDATE: COST, SERVICE, AND STRATEGIC METRICS")
print("=" * 76)
print(f"Candidate: {cycle3_best.name}")
print(f"Total evaluated cost: ${final_solution['total_cost']:,.2f}")
print(f"  Fixed cost: ${final_solution['fixed_cost']:,.2f}")
print(f"  Transportation cost: ${final_solution['transport_cost']:,.2f}")
print(f"Service level: {final_solution['service_level']:.1%}")
print(f"Unmet demand: {final_solution['unmet_demand']:,.1f}")
print(f"Open facilities: {sorted(cycle3_best.open_facilities)}")
print("Facility loads:")
for facility_id, load in sorted(final_solution["facility_loads"].items()):
    capacity = ai_optimizer.facility_by_id[facility_id].capacity
    print(f"  Facility {facility_id}: {load:.1f} / {capacity:.1f} demand units")
print("Strategic indicators:")
print(f"  Mean facility strategic importance: {final_strategic['strategic_importance']:.2f}")
print(f"  Mean expansion potential: {final_strategic['expansion_potential']:.2f}")
print(f"  Importance-weighted risk score: {final_strategic['risk_score']:.2f}")
print(f"  Market-coverage balance: {final_strategic['coverage_balance']:.2f}")
print(f"  Strategic customer coverage: {final_strategic['strategic_coverage']:.2f}")

# Compare against the initial cost-oriented heuristic without implying optimality.
ai_baseline = initial_proposals[0]
ai_baseline_strategic = ai_optimizer.calculate_strategic_metrics(
    ai_baseline.open_facilities
)
cost_change_percent = (
    (final_solution["total_cost"] - ai_baseline.total_cost) / ai_baseline.total_cost * 100
    if ai_baseline.total_cost else 0.0
)
strategic_change_points = (
    final_strategic["strategic_importance"]
    - ai_baseline_strategic["strategic_importance"]
) * 100
risk_change_points = (
    final_strategic["risk_score"] - ai_baseline_strategic["risk_score"]
) * 100

print("\nComparison with the initial cost-oriented heuristic:")
print(f"  Cost change: {cost_change_percent:+.1f}%")
print(f"  Strategic-importance change: {strategic_change_points:+.1f} percentage points")
print(f"  Risk-score change: {risk_change_points:+.1f} percentage points")
print(
    "A positive or negative change is a comparison between two heuristic "
    "candidates; it is not evidence that collaboration improved a global optimum."
)
print(f"Descriptive collaboration score: {decision.collaboration_score:.3f}")

## Interpretation and limitations

The decision cycles illustrate how preference weights can change which of the available candidates a human selects. The proposal heuristics remain fixed across cycles; the preference update changes the human ranking, not the underlying cost calculation or an AI learning model. Likewise, a constraint is a rule applied to facility selection, not a guarantee that the resulting candidate will meet a service target. Always inspect its service level, unmet demand, facility loads, and cost after constraints are applied.

The preference score is a transparent utility index, not a calibrated estimate of business value. It combines normalized cost attractiveness, risk aversion, strategic importance, and expansion potential using the stated weights. The collaboration score is also a descriptive weighted index. Neither score establishes that a candidate is optimal, that human input caused an improvement, or that the model captures every relevant business concern.

### Advantages and limitations

**Potential advantages**
- Domain expertise can express business rules and strategic priorities that are absent from a narrow cost calculation.
- Explanations and separate cost, service, and strategic metrics make candidate trade-offs easier to inspect.
- Explicit constraints make some human requirements operational within the candidate-generation workflow.

**Limitations**
- Greedy assignment and facility-selection heuristics do not guarantee globally optimal facility choices or assignments.
- Customer assignment order can affect the greedy result; indivisible demand and the simplified distance proxy may not match operational reality.
- The preference weights and qualitative thresholds are illustrative, and human judgments can introduce subjectivity or bias.
- The notebook simulates a collaboration workflow; it does not implement a live interface, a validated learning system, or organizational governance.

### When this decision-support pattern may be useful

Human review can be valuable for strategic, high-impact decisions when relevant business context is difficult to encode completely. It is less suitable when decisions must be fully automated, response times are very short, or strict operational guarantees are required. In any application, validate the model assumptions, data, service requirements, and constraints with domain experts before relying on a recommendation.